# Signalement des anomalies d'adresses email — EGE FINESS+

## 1. Imports et connexion

In [1]:
%run ../../config/config.ipynb

import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path

from src.chargement import (
    resoudre_tables, charger_telecoms, charger_ege, charger_pm,
)
from src.email_checker import (
    analyser_emails, marquer_doublons, evaluer_correspondance,
    charger_tlds_iana, charger_bases_noms,
)

pd.set_option("display.max_colwidth", 80)
print("Imports OK")


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Connexion OK
Imports OK


## 2. Référentiels externes — TLD IANA + bases de noms (filtrées par fréquence) + base communes france

In [2]:
DOSSIER_REF = Path("../../data/referentiels")

charger_tlds_iana(cache=DOSSIER_REF / "tlds-alpha-by-domain.txt")

df_prenoms    = pd.read_csv(DOSSIER_REF / "prenom.csv",     sep=",", dtype=str)
df_patronymes = pd.read_csv(DOSSIER_REF / "patronymes.csv", sep=",", dtype=str)

charger_bases_noms(df_prenoms, df_patronymes)

from src.email_checker import PRENOMS, PATRONYMES, TLDS_VALIDES, SEUIL_PRENOM, SEUIL_PATRONYME
print(f"TLD valides                      : {len(TLDS_VALIDES):,}")
print(f"Prénoms retenus (sum >= {SEUIL_PRENOM})     : {len(PRENOMS):,}")
print(f"Patronymes retenus (count >= {SEUIL_PATRONYME}) : {len(PATRONYMES):,}")

TLD valides                      : 1,437
Prénoms retenus (sum >= 500)     : 1,309
Patronymes retenus (count >= 100) : 14,710


In [3]:
df_communes = pd.read_csv(DOSSIER_REF / "communes-france-2026.csv", dtype=str)

from src.email_checker import charger_geo_insee
stats_geo = charger_geo_insee(df_communes, seuil_population=2000)
print(f"Villes retenues (pop >= 2000) : {stats_geo['villes']:,}")
print(f"Départements : {stats_geo['departements']} · Régions : {stats_geo['regions']}")

Villes retenues (pop >= 2000) : 5,200
Départements : 101 · Régions : 18


## 3. Chargement des EGE FINESS+

L'email n'est plus une colonne de la structure : il vient de la table `Telecom`,
atteinte par la liaison contact et pivotée par canal.

In [4]:
TABLES = resoudre_tables(conn)

print("\nTelecoms :")
telecoms = charger_telecoms(conn, TABLES)

print("\nEGE :")
df_ege = charger_ege(conn, TABLES, telecoms)
print(f"\nEGE FINESS actifs chargés : {len(df_ege):,}")

   pm           -> T_FAC_PmSmsse
   ege          -> T_FAC_Ege
   adresse      -> T_FAC_Adresse
   telecom      -> T_FAC_Telecom
   pm_adresse   -> T_FAC_PmSmsseAdresse
   ege_adresse  -> T_FAC_EgeAdresse
   pm_contact   -> T_FAC_PmSmsseContact
   ege_contact  -> T_FAC_EgeContact

Telecoms :
   3 lignes de canal sentinelle ecartees
   324 doublons (contact, canal) : premiere valeur conservee
   telecoms pivotes : 355,904 contacts, colonnes ['TEL_MAIL', 'TEL_TELECOPIE', 'TEL_TELEPHONE']

EGE :
   228,793 lignes brutes
   228,793 lignes -> 104,667 structures, 150 contacts au maximum pour une structure

EGE FINESS actifs chargés : 104,667


## 4. Vue d'ensemble

In [5]:
total      = len(df_ege)
avec_email = df_ege["TEL_MAIL"].notna().sum()
sans_email = total - avec_email
uniques    = df_ege["TEL_MAIL"].nunique()

print("─" * 50)
print("PANORAMA DES EMAILS EGE FINESS")
print("─" * 50)
print(f"  EGE actifs total            : {total:>10,}")
print(f"  Email renseigné             : {avec_email:>10,}  ({avec_email/total*100:.1f} %)")
print(f"  Email absent                : {sans_email:>10,}  ({sans_email/total*100:.1f} %)")
print(f"  Emails uniques              : {uniques:>10,}")
print(f"  Doublons potentiels         : {avec_email - uniques:>10,}")
print("─" * 50)

──────────────────────────────────────────────────
PANORAMA DES EMAILS EGE FINESS
──────────────────────────────────────────────────
  EGE actifs total            :    104,667
  Email renseigné             :     72,075  (68.9 %)
  Email absent                :     32,592  (31.1 %)
  Emails uniques              :     56,301
  Doublons potentiels         :     15,774
──────────────────────────────────────────────────


## 5. Détection des anomalies

In [6]:
print("Analyse des anomalies + classification...")
df = analyser_emails(df_ege, col_email="TEL_MAIL")
df = marquer_doublons(df, col_email_norm="email_norm", col_id="NB_EgeId")
print(f"Analyse terminée — {len(df):,} EGE traités.")

Analyse des anomalies + classification...
Analyse terminée — 104,667 EGE traités.


## 6. Correspondance partie locale ↔ raison sociale / adresse

La dénomination de la personne morale parente est récupérée sur `NB_PmSmsseId`,
un identifiant entier porté par l'établissement.

In [7]:
df_pm_ref = charger_pm(conn, TABLES, telecoms)
map_raison_pm = dict(zip(df_pm_ref["NB_PmSmsseId"].astype(str),
                         df_pm_ref["TX_DenominationPm"].astype(str)))
print(f"Dénominations PM chargées : {len(map_raison_pm):,}")

def _corresp_ligne(row):
    raison_pm = map_raison_pm.get(str(row["NB_PmSmsseId"]), "")
    return evaluer_correspondance(
        local_part=row["local_part"],
        raison_principale=row["TX_NomEgeLong"],
        raison_parent=raison_pm,
        adresse=row.get("TX_LibelleVoie", ""),
        libelle_principal="(EGE)",
        libelle_parent="(PM parente)",
    )

df["correspondance"] = ""
masque = df["local_part"] != ""
df.loc[masque, "correspondance"] = df.loc[masque].apply(_corresp_ligne, axis=1)

print("\nRépartition des correspondances :")
print(df[df["correspondance"] != ""]["correspondance"].value_counts().to_string())

   84,015 lignes brutes
   84,015 lignes -> 56,194 structures, 3 contacts au maximum pour une structure
Dénominations PM chargées : 56,194

Répartition des correspondances :
correspondance
Aucune correspondance                    41173
Raison sociale (EGE)                     25968
Raison sociale (PM parente)               2803
Raison sociale (EGE) + Adresse            1763
Adresse                                    322
Raison sociale (PM parente) + Adresse       46


In [8]:
from src.email_checker import detecter_geo

geo = df.apply(lambda row: detecter_geo(row["local_part"], row.get("TX_CogCommune")), axis=1)
df["geo_local"]      = geo.apply(lambda x: x["geo_local"])
df["geo_concordant"] = geo.apply(lambda x: x["geo_concordant"])

apercu = df[(df["geo_local"] != "") & (df["geo_concordant"] == False)]
print(f"Emails portant un lieu : {(df['geo_local'] != '').sum():,}")
print(f"  dont concordants     : {(df['geo_concordant'] == True).sum():,}")
print(f"  dont NON concordants : {len(apercu):,}")
apercu[["TX_NomEgeLong", "TEL_MAIL", "geo_local"]].head(10)

Emails portant un lieu : 8,882
  dont concordants     : 7,036
  dont NON concordants : 1,846


,TX_NomEgeLong,TEL_MAIL,geo_local
0,SAAD APEF NARBONNE,narbonne@apef-services.fr,Ville (arbonne)
80,EHPAD MR NOTRE DAME DU DIMANCHE,notredamedudimanche@orange.fr,Département (manche)
120,EHPAD MONTPLAISIR,montplaisir2@wanadoo.fr,Ville (plaisir)
126,CENTRE DE SOINS INFIRMIERS FILIERIS DE GRAISSESSAC,oeuvres.ales@filieris.fr,Ville (ales)
133,EHPAD LE CHATEAU DE LA ROCHE,chateau.de.la.roche@orange.fr,Ville (roche)
194,SAAD O2 NARBONNE,narbonne@o2.fr,Ville (arbonne)
198,ALEOS,rhone@aleos.asso.fr,Département (rhone)
365,EHPAD RESIDENCE LES JARDINS DE FLORE,lesjardinsdeflore@aliceadsl.fr,Ville (jardin)
406,MAISON D'ACCUEIL SPECIALISEE SAINT VITAL,mas.saintvital@orange.fr,Ville (saintvit)
734,PHARMACIE NEVET - ZOBOUYAN,corbiere.nevet@free.fr,Ville (corbie)


## 7. Distribution globale des signalements

In [9]:
dist_niveau = df["niveau_anomalie"].value_counts().sort_index()
labels = {0: "Valides", 1: "Critiques (niveau 1)", 2: "Qualité (niveau 2)"}

print("─" * 55)
print("DISTRIBUTION GLOBALE DES SIGNALEMENTS")
print("─" * 55)
for niveau, n in dist_niveau.items():
    print(f"  {labels.get(niveau, f'Niveau {niveau}'):<30} : {n:>8,}  ({n/len(df)*100:5.1f} %)")
print("─" * 55)

print("\nDÉTAIL PAR TYPE D'ANOMALIE")
print("─" * 55)
detail = (
    df[df["niveau_anomalie"] > 0]
    .groupby(["niveau_anomalie", "code_anomalie"])
    .size().reset_index(name="nb")
    .sort_values(["niveau_anomalie", "nb"], ascending=[True, False])
)
for _, r in detail.iterrows():
    print(f"  [{r['niveau_anomalie']}] {r['code_anomalie']:<22} : {r['nb']:>6,}")

───────────────────────────────────────────────────────
DISTRIBUTION GLOBALE DES SIGNALEMENTS
───────────────────────────────────────────────────────
  Valides                        :   30,926  ( 29.5 %)
  Critiques (niveau 1)           :   32,727  ( 31.3 %)
  Qualité (niveau 2)             :   41,014  ( 39.2 %)
───────────────────────────────────────────────────────

DÉTAIL PAR TYPE D'ANOMALIE
───────────────────────────────────────────────────────
  [1] EMAIL_VIDE             : 32,592
  [1] DOMAINE_SANS_POINT     :     36
  [1] EXTENSION_VIDE         :     22
  [1] TLD_INEXISTANT         :     21
  [1] ESPACE                 :     18
  [1] DOMAINE_INEXISTANT     :     16
  [1] EXTENSION_TYPO         :     10
  [1] MULTI_AROBAS           :      6
  [1] EXTENSION_LIEU         :      2
  [1] CARACTERE_INTERDIT     :      1
  [1] DOMAINE_VIDE           :      1
  [1] EXTENSION_FICHIER      :      1
  [1] POINTS_CONSECUTIFS     :      1
  [2] DOUBLON                : 21,106
  [2] EMAIL_G

## 8. Classification de la partie locale

In [10]:
classif = df[df["type_local"] != ""]
print(f"Classification de {len(classif):,} emails\n")

print("Partie locale :")
print(classif["type_local"].value_counts().to_string())
print("\nDomaine :")
print(classif["type_domaine"].value_counts().to_string())
print("\nCroisement local x domaine :")
print(pd.crosstab(classif["type_local"], classif["type_domaine"]).to_string())

Classification de 71,933 emails

Partie locale :
type_local
INDETERMINE          24612
GENERIQUE            21142
INSTITUTIONNEL       16268
NOMINATIF             6843
NOMINATIF_PARTIEL     3068

Domaine :
type_domaine
PROPRE    52025
PUBLIC    19908

Croisement local x domaine :
type_domaine       PROPRE  PUBLIC
type_local                       
GENERIQUE           19479    1663
INDETERMINE         16096    8516
INSTITUTIONNEL       8529    7739
NOMINATIF            5406    1437
NOMINATIF_PARTIEL    2515     553


## 9. Anomalies critiques (niveau 1)

In [11]:
df_n1 = df[df["niveau_anomalie"] == 1].copy()
print(f"Total anomalies critiques : {len(df_n1):,}\n")

COLS_AFFICH = [
    "TX_NumFinessEge", "TX_NomEgeLong", "TEL_MAIL",
    "code_anomalie", "libelle_anomalie", "suggestion",
]
cols = [c for c in COLS_AFFICH if c in df_n1.columns]

for code_a, groupe in df_n1.groupby("code_anomalie"):
    print(f"\n{'─'*60}")
    print(f"  {code_a}  ({len(groupe):,} cas)")
    print(f"  {groupe['libelle_anomalie'].iloc[0]}")
    print(f"{'─'*60}")
    print(groupe[cols].head(5).to_string(index=False))

Total anomalies critiques : 32,727


────────────────────────────────────────────────────────────
  CARACTERE_INTERDIT  (1 cas)
  Caractère(s) interdit(s) dans l'adresse : ,
────────────────────────────────────────────────────────────
TX_NumFinessEge          TX_NomEgeLong         TEL_MAIL      code_anomalie                            libelle_anomalie suggestion
      280502741 A.I.R.B.P IRC CHARTRES airbp@wanadoo,fr CARACTERE_INTERDIT Caractère(s) interdit(s) dans l'adresse : ,           

────────────────────────────────────────────────────────────
  DOMAINE_INEXISTANT  (16 cas)
  Le domaine « @gmail.fr » n'existe pas — variante invalide d'un service connu (ex : gmail.fr → gmail.com)
────────────────────────────────────────────────────────────
TX_NumFinessEge                    TX_NomEgeLong                         TEL_MAIL      code_anomalie                                                                                         libelle_anomalie suggestion
      400009072            

## 10. Signalements qualité (niveau 2)

In [12]:
df_n2 = df[df["niveau_anomalie"] == 2].copy()
print(f"Total signalements qualité : {len(df_n2):,}\n")

for code_a, groupe in df_n2.groupby("code_anomalie"):
    print(f"\n{'─'*60}")
    print(f"  {code_a}  ({len(groupe):,} cas)")
    print(f"  {groupe['libelle_anomalie'].iloc[0]}")
    print(f"{'─'*60}")
    print(groupe[cols].head(5).to_string(index=False))

Total signalements qualité : 41,014


────────────────────────────────────────────────────────────
  DOUBLON  (21,106 cas)
  Email partagé par 4.0 structures distinctes — vérifier si intentionnel
────────────────────────────────────────────────────────────
TX_NumFinessEge                                         TX_NomEgeLong                         TEL_MAIL code_anomalie                                                       libelle_anomalie suggestion
      680007861              CENTRE DE LUTTE ANTITUBERCULEUSE CEA SUD                 clat68@alsace.eu       DOUBLON Email partagé par 4.0 structures distinctes — vérifier si intentionnel           
      110008190                          SAD AMELIS DOMICILE SERVICES          contact@domicilplus.com       DOUBLON Email partagé par 3.0 structures distinctes — vérifier si intentionnel           
      340784081   CENTRE HEBERGEMENT ET READAPTATION SOCIALE A.B.E.S.              direction@abes34.fr       DOUBLON Email partagé par 2.0 structu

## 11. Emails grand public — top domaines

In [13]:
df_gp = df_n2[df_n2["code_anomalie"] == "EMAIL_GRAND_PUBLIC"]

if len(df_gp) > 0:
    top_dom = df_gp["domaine"].value_counts().head(20)
    print(f"Top 20 domaines grand public ({len(df_gp):,} cas total) :\n")
    for domaine, n in top_dom.items():
        barre = "█" * min(int(n / top_dom.max() * 30), 30)
        print(f"  @{domaine:<28} {n:>6,}  {barre}")
else:
    print("Aucun email grand public détecté.")

Top 20 domaines grand public (19,908 cas total) :

  @gmail.com                     7,882  ██████████████████████████████
  @orange.fr                     4,890  ██████████████████
  @wanadoo.fr                    4,160  ███████████████
  @hotmail.fr                      641  ██
  @yahoo.fr                        624  ██
  @free.fr                         318  █
  @hotmail.com                     306  █
  @outlook.fr                      242  
  @laposte.net                     229  
  @yahoo.com                       111  
  @sfr.fr                           97  
  @live.fr                          84  
  @outlook.com                      54  
  @club-internet.fr                 49  
  @neuf.fr                          47  
  @icloud.com                       40  
  @bbox.fr                          39  
  @protonmail.com                   20  
  @me.com                           17  
  @aliceadsl.fr                     13  


## 12. Synthèse finale

In [14]:
n_total    = len(df)
n_vides    = (df["code_anomalie"] == "EMAIL_VIDE").sum()
n_renseign = max(n_total - n_vides, 1)
n_critique = (df["niveau_anomalie"] == 1).sum()
n_qualite  = (df["niveau_anomalie"] == 2).sum()
n_valides  = (df["niveau_anomalie"] == 0).sum()

print("=" * 60)
print("SYNTHÈSE GLOBALE — SIGNALEMENT EMAIL EGE FINESS")
print("=" * 60)
print(f"  EGE actifs analysés         : {n_total:>8,}")
print(f"  Email absent (vide)         : {n_vides:>8,}  ({n_vides/n_total*100:.1f} %)")
print()
print(f"  ── Sur les {n_renseign:,} emails renseignés ──")
print(f"  [1] Anomalies critiques     : {n_critique:>8,}")
print(f"  [2] Signalements qualité    : {n_qualite:>8,}")
print(f"  [0] Emails valides          : {n_valides:>8,}")
print("=" * 60)

SYNTHÈSE GLOBALE — SIGNALEMENT EMAIL EGE FINESS
  EGE actifs analysés         :  104,667
  Email absent (vide)         :   32,592  (31.1 %)

  ── Sur les 72,075 emails renseignés ──
  [1] Anomalies critiques     :   32,727
  [2] Signalements qualité    :   41,014
  [0] Emails valides          :   30,926


## 13. Export Excel — Rapport de signalement

In [15]:
from openpyxl.styles import PatternFill, Font, Alignment

DOSSIER_SORTIE = Path("../../results/email")
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
FICHIER_SORTIE = DOSSIER_SORTIE / "signalement_emails_ege.xlsx"

COLS_EXPORT = [
    "NB_EgeId", "TX_NumFinessEge", "NB_PmSmsseId",
    "TX_NomEgeLong", "TX_CogCommune", "departement", "adresse_complete",
    "TEL_MAIL", "email_norm",
    "code_anomalie", "libelle_anomalie", "suggestion",
    "type_local", "type_domaine", "correspondance",
    "geo_local", "geo_concordant",
]
cols_dispo = [c for c in COLS_EXPORT if c in df.columns]

def style_entete(ws, couleur):
    for cell in ws[1]:
        cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
        cell.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 28

def remplir(ws, couleur):
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
            cell.font = Font(name="Arial", size=9)

def auto_width(ws, max_w=50):
    for col in ws.columns:
        w = max((len(str(c.value or "")) for c in col), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(w + 3, max_w)

df_vides     = df[df["code_anomalie"] == "EMAIL_VIDE"][cols_dispo]
df_critiques = df[(df["niveau_anomalie"] == 1) & (df["code_anomalie"] != "EMAIL_VIDE")][cols_dispo]
df_doublons  = df[df["code_anomalie"] == "DOUBLON"][cols_dispo]
df_grandpub  = df[df["code_anomalie"] == "EMAIL_GRAND_PUBLIC"][cols_dispo]
df_valides   = df[df["niveau_anomalie"] == 0][cols_dispo]

df_nominatifs   = df[df["type_local"].isin(["NOMINATIF", "NOMINATIF_PARTIEL"])][cols_dispo]
df_indetermines = df[df["type_local"] == "INDETERMINE"][cols_dispo]

df_synth = pd.DataFrame([
    {"Catégorie": "EGE actifs total",     "Nombre": n_total,           "Part": "100 %"},
    {"Catégorie": "Email absent (vide)",         "Nombre": n_vides,           "Part": f"{n_vides/n_total*100:.1f} %"},
    {"Catégorie": "—",                           "Nombre": "",                "Part": ""},
    {"Catégorie": "[1] Anomalies critiques",     "Nombre": len(df_critiques), "Part": f"{len(df_critiques)/n_renseign*100:.1f} %"},
    {"Catégorie": "[2] Doublons",                "Nombre": len(df_doublons),  "Part": f"{len(df_doublons)/n_renseign*100:.1f} %"},
    {"Catégorie": "[2] Grand public",            "Nombre": len(df_grandpub),  "Part": f"{len(df_grandpub)/n_renseign*100:.1f} %"},
    {"Catégorie": "[0] Emails valides",          "Nombre": len(df_valides),   "Part": f"{len(df_valides)/n_renseign*100:.1f} %"},
    {"Catégorie": "—",                           "Nombre": "",                "Part": ""},
    {"Catégorie": "Vue croisée (hors total ci-dessus, quel que soit le domaine)", "Nombre": "", "Part": ""},
    {"Catégorie": "Nominatifs (NOMINATIF + PARTIEL)", "Nombre": len(df_nominatifs),   "Part": f"{len(df_nominatifs)/n_renseign*100:.1f} %"},
    {"Catégorie": "Indéterminés",                     "Nombre": len(df_indetermines), "Part": f"{len(df_indetermines)/n_renseign*100:.1f} %"},
])

with pd.ExcelWriter(FICHIER_SORTIE, engine="openpyxl") as writer:
    df_synth.to_excel(writer, sheet_name="Synthèse", index=False)
    style_entete(writer.sheets["Synthèse"], "1F3864")
    auto_width(writer.sheets["Synthèse"], max_w=40)

    for nom_f, donnees, ent, fond in [
        ("Emails_Vides",        df_vides,        "C62828", "F8CECC"),
        ("Anomalies_Critiques", df_critiques,    "C62828", "F8CECC"),
        ("Doublons",            df_doublons,     "F57C00", "FFF2CC"),
        ("Grand_Public",        df_grandpub,     "F57C00", "FFF2CC"),
        ("Emails_Valides",      df_valides,      "2E7D32", "EBF5EB"),
        ("Nominatifs",          df_nominatifs,   "1565C0", "D6E4F0"),
        ("Indetermines",        df_indetermines, "6A1B9A", "E8DAEF"),
    ]:
        if len(donnees) > 0:
            donnees.to_excel(writer, sheet_name=nom_f, index=False)
            style_entete(writer.sheets[nom_f], ent)
            remplir(writer.sheets[nom_f], fond)
            auto_width(writer.sheets[nom_f])

print(f"Export OK → {FICHIER_SORTIE.resolve()}")
print("\nFeuilles produites :")
for nom_f, donnees in [("Emails_Vides", df_vides), ("Anomalies_Critiques", df_critiques),
                       ("Doublons", df_doublons), ("Grand_Public", df_grandpub),
                       ("Emails_Valides", df_valides), ("Nominatifs", df_nominatifs),
                       ("Indetermines", df_indetermines)]:
    print(f"  {nom_f:<22} : {len(donnees):,}")

Export OK → /home/jovyan/work/finess+_refonte/signalement_data_contact_finess+/results/email/signalement_emails_ege.xlsx

Feuilles produites :
  Emails_Vides           : 32,592
  Anomalies_Critiques    : 135
  Doublons               : 21,106
  Grand_Public           : 19,908
  Emails_Valides         : 30,926
  Nominatifs             : 9,911
  Indetermines           : 24,612
